# House Price Prediction – King County, USA

Final project of the IBM course *Data Analysis with Python* (IBM Data Analyst Professional Certificate).

**Goal:** estimate the market price of a house from its features (living area, grade, floors, waterfront view, location and more) for a real estate investment trust.

**Author:** Emmanuel Ogochukwu Okatta

## Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
%matplotlib inline

## Module 1: Importing the data

In [ ]:
filepath = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DA0101EN-SkillsNetwork/labs/FinalModule_Coursera/data/kc_house_data_NaN.csv'
df = pd.read_csv(filepath)
df.head()

### Question 1 – Data types of each column

In [ ]:
df.dtypes

## Module 2: Data wrangling

### Question 2 – Drop `id` and `Unnamed: 0`, then get a statistical summary

In [ ]:
df.drop(["id", "Unnamed: 0"], axis=1, inplace=True)
df.describe()

`bedrooms` and `bathrooms` have missing values. They are replaced with the column mean.

In [ ]:
print("number of NaN values for the column bedrooms :", df['bedrooms'].isnull().sum())
print("number of NaN values for the column bathrooms :", df['bathrooms'].isnull().sum())

In [ ]:
df['bedrooms'] = df['bedrooms'].replace(np.nan, df['bedrooms'].mean())
df['bathrooms'] = df['bathrooms'].replace(np.nan, df['bathrooms'].mean())

print("number of NaN values for the column bedrooms :", df['bedrooms'].isnull().sum())
print("number of NaN values for the column bathrooms :", df['bathrooms'].isnull().sum())

## Module 3: Exploratory data analysis

### Question 3 – Number of houses by number of floors

In [ ]:
df['floors'].value_counts().to_frame()

### Question 4 – Price outliers: houses with and without a waterfront view

In [ ]:
sns.boxplot(x="waterfront", y="price", data=df)
plt.show()

### Question 5 – Relationship between `sqft_above` and price

In [ ]:
sns.regplot(x="sqft_above", y="price", data=df, line_kws={"color": "red"})
plt.ylim(0,)
plt.show()

Correlation of all numeric features with price:

In [ ]:
df.corr(numeric_only=True)['price'].sort_values()

## Module 4: Model development

In [ ]:
X = df[['long']]
Y = df['price']
lm = LinearRegression()
lm.fit(X, Y)
lm.score(X, Y)

### Question 6 – Linear regression with `sqft_living`

In [ ]:
X = df[['sqft_living']]
Y = df['price']
lm = LinearRegression()
lm.fit(X, Y)
print(lm.score(X, Y))

### Question 7 – Multiple linear regression

In [ ]:
features = ["floors", "waterfront", "lat", "bedrooms", "sqft_basement", "view",
            "bathrooms", "sqft_living15", "sqft_above", "grade", "sqft_living"]
Z = df[features]
lm.fit(Z, Y)
print(lm.score(Z, Y))

### Question 8 – Pipeline: scaling, polynomial features and linear regression

In [ ]:
Input = [('scale', StandardScaler()),
         ('polynomial', PolynomialFeatures(include_bias=False)),
         ('model', LinearRegression())]
pipe = Pipeline(Input)
Z = Z.astype(float)
pipe.fit(Z, Y)
ypipe = pipe.predict(Z)
print(r2_score(Y, ypipe))

## Module 5: Model evaluation and refinement

In [ ]:
X = df[features]
Y = df['price']
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.15, random_state=1)
print("number of test samples:", x_test.shape[0])
print("number of training samples:", x_train.shape[0])

### Question 9 – Ridge regression (alpha = 0.1), evaluated on the test data

In [ ]:
RidgeModel = Ridge(alpha=0.1)
RidgeModel.fit(x_train, y_train)
yhat = RidgeModel.predict(x_test)
print(r2_score(y_test, yhat))

### Question 10 – Second-order polynomial transform + Ridge regression

In [ ]:
pr = PolynomialFeatures(degree=2)
x_train_pr = pr.fit_transform(x_train)
x_test_pr = pr.transform(x_test)
RidgeModel = Ridge(alpha=0.1)
RidgeModel.fit(x_train_pr, y_train)
y_hat = RidgeModel.predict(x_test_pr)
print(r2_score(y_test, y_hat))

## Conclusion

- Living area and grade are the strongest price drivers. A single feature explains only about half of the price variation (R² ≈ 0.49).
- Using 11 features raises R² to about 0.66, and adding polynomial features raises it to about 0.75, so the relationships are not purely linear.
- On unseen test data, polynomial features with Ridge regression reach R² ≈ 0.70.
- Next steps: log-transform the price, investigate outliers (e.g. a house with 33 bedrooms), tune alpha with cross-validation and add location features such as `zipcode`.